In [2]:
import sqlite3
from pathlib import Path
import pandas as pd

DB_PATH = Path("sql_notes_demo.db")


def create_connection(db_path: Path = DB_PATH):
    return sqlite3.connect(db_path)


def reset_database(db_path: Path = DB_PATH):
    if db_path.exists():
        db_path.unlink()


def create_tables(conn):
    cursor = conn.cursor()
    # FOREIGN KEY (category_id) REFERENCES categories(category_id)
    # ADD IN products
    schema_sql = """
    PRAGMA foreign_keys = ON;

    CREATE TABLE customers (
        customer_id INTEGER PRIMARY KEY,
        first_name TEXT NOT NULL,
        last_name TEXT NOT NULL,
        city TEXT NOT NULL,
        country TEXT NOT NULL,
        email TEXT UNIQUE,
        signup_date TEXT NOT NULL
    );

    CREATE TABLE employees (
        employee_id INTEGER PRIMARY KEY,
        first_name TEXT NOT NULL,
        last_name TEXT NOT NULL,
        country TEXT NOT NULL,
        department TEXT NOT NULL,
        hire_date TEXT NOT NULL
    );

    CREATE TABLE categories (
        category_id INTEGER PRIMARY KEY,
        category_name TEXT NOT NULL UNIQUE
    );

    CREATE TABLE products (
        product_id INTEGER PRIMARY KEY,
        product_name TEXT NOT NULL,
        category_id INTEGER NOT NULL,
        unit_price REAL NOT NULL,
        stock_quantity INTEGER NOT NULL,
        active INTEGER NOT NULL DEFAULT 1
    );

    CREATE TABLE orders (
        order_id INTEGER PRIMARY KEY,
        customer_id INTEGER NOT NULL,
        employee_id INTEGER,
        order_date TEXT NOT NULL,
        status TEXT NOT NULL,
        FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
        FOREIGN KEY (employee_id) REFERENCES employees(employee_id)
    );

    CREATE TABLE order_items (
        order_item_id INTEGER PRIMARY KEY,
        order_id INTEGER NOT NULL,
        product_id INTEGER NOT NULL,
        quantity INTEGER NOT NULL,
        unit_price REAL NOT NULL,
        discount REAL NOT NULL DEFAULT 0,
        FOREIGN KEY (order_id) REFERENCES orders(order_id),
        FOREIGN KEY (product_id) REFERENCES products(product_id)
    );
    """
    cursor.executescript(schema_sql)
    conn.commit()


def insert_sample_data(conn):
    cursor = conn.cursor()

    customers = [
        (1, "Alice", "Martin", "Geneva", "Switzerland", "alice.martin@email.com", "2024-01-15"),
        (2, "Karim", "Benali", "Lausanne", "Switzerland", "karim.benali@email.com", "2024-02-03"),
        (3, "Sofia", "Rossi", "Milan", "Italy", "sofia.rossi@email.com", "2024-03-11"),
        (4, "Lucas", "Dubois", "Lyon", "France", "lucas.dubois@email.com", "2024-03-25"),
        (5, "Emma", "Schneider", "Zurich", "Switzerland", "emma.schneider@email.com", "2024-04-01"),
        (6, "Noah", "Garcia", "Barcelona", "Spain", "noah.garcia@email.com", "2024-04-18"),
        (7, "Maya", "Haddad", "Geneva", "Switzerland", "maya.haddad@email.com", "2024-05-07"),
        (8, "Leo", "Moreau", "Paris", "France", "leo.moreau@email.com", "2024-05-21")
    ]

    employees = [
        (1, "Sarah", "Dupont", "Switzerland", "Sales", "2023-01-10"),
        (2, "Nicolas", "Weber", "Algeria", "Sales", "2023-06-05"),
        (3, "Ines", "Favre", "France", "Support", "2024-02-12")
    ]

    categories = [
        (1, "Beverages"),
        (2, "Snacks"),
        (3, "Electronics"),
        (4, "Books"),
        (8, "Test")
    ]

    products = [
        (1, "Coffee Beans 1kg", 10, 18.50, 120, 1),
        (2, "Green Tea Box", 11, 7.90, 85, 1),
        (3, "Protein Bar", 12, 2.50, 300, 1),
        (4, "Potato Chips", 2, 2.50, 150, 1),
        (5, "Wireless Mouse", 3, 24.90, 40, 1),
        (6, "Mechanical Keyboard", 3, 79.00, 25, 1),
        (7, "USB-C Cable", 3, 9.90, 200, 1),
        (8, "SQL for Beginners", 4, 29.90, 60, 1),
        (9, "Advanced Data Modeling", 4, 44.00, 20, 1),
        (10, "Old Headphones", 3, 15.00, 0, 0)
    ]

    orders = [
        (1, 1, 1, "2025-01-10", "shipped"),
        (2, 2, 2, "2025-01-12", "pending"),
        (3, 1, 1, "2025-01-18", "delivered"),
        (4, 3, 2, "2025-02-02", "cancelled"),
        (5, 4, 1, "2025-02-08", "delivered"),
        (6, 5, 3, "2025-02-18", "shipped"),
        (7, 6, 2, "2025-03-01", "pending"),
        (8, 7, 1, "2025-03-03", "delivered"),
        (9, 7, 3, "2025-03-05", "pending"),
        (10, 2, 2, "2025-03-07", "delivered")
    ]

    order_items = [
        (1, 1, 1, 2, 18.50, 0.00),
        (2, 1, 3, 6, 2.50, 0.10),
        (3, 2, 5, 1, 24.90, 0.00),
        (4, 2, 7, 2, 9.90, 0.00),
        (5, 3, 8, 1, 29.90, 0.00),
        (6, 3, 2, 3, 7.90, 0.05),
        (7, 4, 6, 1, 79.00, 0.00),
        (8, 5, 4, 4, 2.50, 0.00),
        (9, 5, 3, 10, 2.50, 0.15),
        (10, 6, 9, 1, 44.00, 0.00),
        (11, 6, 1, 1, 18.50, 0.00),
        (12, 7, 5, 2, 24.90, 0.20),
        (13, 7, 7, 3, 9.90, 0.00),
        (14, 8, 8, 2, 29.90, 0.10),
        (15, 8, 3, 5, 2.50, 0.00),
        (16, 9, 2, 4, 7.90, 0.00),
        (17, 9, 4, 2, 2.50, 0.00),
        (18, 10, 6, 1, 79.00, 0.05),
        (19, 10, 1, 1, 18.50, 0.00)
    ]

    cursor.executemany("INSERT INTO customers VALUES (?, ?, ?, ?, ?, ?, ?)", customers)
    cursor.executemany("INSERT INTO employees VALUES (?, ?, ?, ?, ?, ?)", employees)
    cursor.executemany("INSERT INTO categories VALUES (?, ?)", categories)
    cursor.executemany("INSERT INTO products VALUES (?, ?, ?, ?, ?, ?)", products)
    cursor.executemany("INSERT INTO orders VALUES (?, ?, ?, ?, ?)", orders)
    cursor.executemany("INSERT INTO order_items VALUES (?, ?, ?, ?, ?, ?)", order_items)

    conn.commit()


def setup_database(db_path: Path = DB_PATH):
    reset_database(db_path)
    conn = create_connection(db_path)
    create_tables(conn)
    insert_sample_data(conn)
    return conn

def run_query(sql: str):
    return pd.read_sql_query(sql, conn)

def show_table(table_name: str):
    return pd.read_sql_query(f"SELECT * FROM {table_name}", conn)


conn = setup_database()



# 1. SELECT
**`SELECT`** is used to retrieve data from one or more columns of a table

In [3]:
# Selecting specific columns
query = ("""
SELECT first_name, last_name, city
FROM customers;
""")
run_query(query)

,first_name,last_name,city
0,Alice,Martin,Geneva
1,Karim,Benali,Lausanne
2,Sofia,Rossi,Milan
3,Lucas,Dubois,Lyon
4,Emma,Schneider,Zurich
5,Noah,Garcia,Barcelona
6,Maya,Haddad,Geneva
7,Leo,Moreau,Paris


In [4]:
#  Selecting all columns
query = ("""
SELECT *
FROM customers;
""")
run_query(query)

,customer_id,first_name,last_name,city,country,email,signup_date
0,1,Alice,Martin,Geneva,Switzerland,alice.martin@email.com,2024-01-15
1,2,Karim,Benali,Lausanne,Switzerland,karim.benali@email.com,2024-02-03
2,3,Sofia,Rossi,Milan,Italy,sofia.rossi@email.com,2024-03-11
3,4,Lucas,Dubois,Lyon,France,lucas.dubois@email.com,2024-03-25
4,5,Emma,Schneider,Zurich,Switzerland,emma.schneider@email.com,2024-04-01
5,6,Noah,Garcia,Barcelona,Spain,noah.garcia@email.com,2024-04-18
6,7,Maya,Haddad,Geneva,Switzerland,maya.haddad@email.com,2024-05-07
7,8,Leo,Moreau,Paris,France,leo.moreau@email.com,2024-05-21


In [5]:
# Removing duplicates with DISTINCT (applies to the full combination of selected columns)
query = ("""
SELECT DISTINCT country
FROM customers;
""")
run_query(query)

,country
0,Switzerland
1,Italy
2,France
3,Spain


In [6]:
# DISTINCT result will change with multiple columns (distinct combination of all columns)
query = ("""
SELECT DISTINCT country, city
FROM customers;
""")
run_query(query)

,country,city
0,Switzerland,Geneva
1,Switzerland,Lausanne
2,Italy,Milan
3,France,Lyon
4,Switzerland,Zurich
5,Spain,Barcelona
6,France,Paris


In [7]:
# Expressions can be used into a SELECT (arithmetic expression)
query = ("""
SELECT order_id, quantity, unit_price, quantity * unit_price
FROM order_items;
""")
run_query(query)

,order_id,quantity,unit_price,quantity * unit_price
0,1,2,18.5,37.0
1,1,6,2.5,15.0
2,2,1,24.9,24.9
3,2,2,9.9,19.8
4,3,1,29.9,29.9
5,3,3,7.9,23.7
6,4,1,79.0,79.0
7,5,4,2.5,10.0
8,5,10,2.5,25.0
9,6,1,44.0,44.0


In [8]:
# Expressions can be used into a SELECT statement (text expression)
query = ("""
SELECT first_name || ' ' || last_name
FROM customers;
""")
run_query(query)

,first_name || ' ' || last_name
0,Alice Martin
1,Karim Benali
2,Sofia Rossi
3,Lucas Dubois
4,Emma Schneider
5,Noah Garcia
6,Maya Haddad
7,Leo Moreau


In [9]:
# Alias can be used for a column in a SELECT statement (AS is optionnel, but it's better to put it for readability)
query = ("""
SELECT first_name || ' ' || last_name AS [Full name]
FROM customers;
""")
run_query(query)

# [] is only if we want spaces
# SELECT first_name || ' ' || last_name AS full_name

,Full name
0,Alice Martin
1,Karim Benali
2,Sofia Rossi
3,Lucas Dubois
4,Emma Schneider
5,Noah Garcia
6,Maya Haddad
7,Leo Moreau


In [10]:
# Constants (fixed values) can be used in a SELECT statement
query = ("""
SELECT 'employee' AS role, first_name || ' ' || last_name AS [Full name]
FROM customers;
""")
run_query(query)

,role,Full name
0,employee,Alice Martin
1,employee,Karim Benali
2,employee,Sofia Rossi
3,employee,Lucas Dubois
4,employee,Emma Schneider
5,employee,Noah Garcia
6,employee,Maya Haddad
7,employee,Leo Moreau


In [11]:
# LIMIT to limit the number of records to return
query = ("""
SELECT *
FROM customers
LIMIT 3;
""")
run_query(query)

# SELECT TOP N * FROM ... (SQL SERVER / MS ACCESS)
# SELECT * FROM ... FETCH FIRST n ROWS ONLY (ORACLE)

,customer_id,first_name,last_name,city,country,email,signup_date
0,1,Alice,Martin,Geneva,Switzerland,alice.martin@email.com,2024-01-15
1,2,Karim,Benali,Lausanne,Switzerland,karim.benali@email.com,2024-02-03
2,3,Sofia,Rossi,Milan,Italy,sofia.rossi@email.com,2024-03-11


In [12]:
# Select the top N records with a condition and in a certain order (useful if you want the 3 best ...)
query = ("""
SELECT *
FROM customers
WHERE country = 'Switzerland'
ORDER BY first_name
LIMIT 3
""")
run_query(query)

,customer_id,first_name,last_name,city,country,email,signup_date
0,1,Alice,Martin,Geneva,Switzerland,alice.martin@email.com,2024-01-15
1,5,Emma,Schneider,Zurich,Switzerland,emma.schneider@email.com,2024-04-01
2,2,Karim,Benali,Lausanne,Switzerland,karim.benali@email.com,2024-02-03


# 2. WHERE
The **`WHERE`** clause is used to filter rows in a query (return only rows that satisfy a given condition)

In [13]:
# WHERE is used to filter record with conditions
query = ("""
SELECT *
FROM customers
WHERE Country = 'France'
""")
run_query(query)

,customer_id,first_name,last_name,city,country,email,signup_date
0,4,Lucas,Dubois,Lyon,France,lucas.dubois@email.com,2024-03-25
1,8,Leo,Moreau,Paris,France,leo.moreau@email.com,2024-05-21


In [14]:
# Different operators are avalaible
query = ("""
SELECT *
FROM customers
WHERE customer_id > 4;
""")
run_query(query)

,customer_id,first_name,last_name,city,country,email,signup_date
0,5,Emma,Schneider,Zurich,Switzerland,emma.schneider@email.com,2024-04-01
1,6,Noah,Garcia,Barcelona,Spain,noah.garcia@email.com,2024-04-18
2,7,Maya,Haddad,Geneva,Switzerland,maya.haddad@email.com,2024-05-07
3,8,Leo,Moreau,Paris,France,leo.moreau@email.com,2024-05-21


In [15]:
# NOT is used to return all the records that don't match the specified criteria (reversed a condition)
query = ("""
SELECT *
FROM Customers
WHERE NOT country = 'Switzerland';
""")
run_query(query)

,customer_id,first_name,last_name,city,country,email,signup_date
0,3,Sofia,Rossi,Milan,Italy,sofia.rossi@email.com,2024-03-11
1,4,Lucas,Dubois,Lyon,France,lucas.dubois@email.com,2024-03-25
2,6,Noah,Garcia,Barcelona,Spain,noah.garcia@email.com,2024-04-18
3,8,Leo,Moreau,Paris,France,leo.moreau@email.com,2024-05-21


In [16]:
# Logical operators to combine multiple conditions (AND)
query = ("""
SELECT *
FROM Customers
WHERE country = 'Switzerland'
    AND city = 'Geneva'
    AND customer_id > 6;
""")
run_query(query)

,customer_id,first_name,last_name,city,country,email,signup_date
0,7,Maya,Haddad,Geneva,Switzerland,maya.haddad@email.com,2024-05-07


In [17]:
# Logical operators to combine multiple conditions (OR)
query = ("""
SELECT *
FROM Customers
WHERE country = 'Switzerland'
    OR country = 'France'
    OR customer_id > 6;
""")
run_query(query)

,customer_id,first_name,last_name,city,country,email,signup_date
0,1,Alice,Martin,Geneva,Switzerland,alice.martin@email.com,2024-01-15
1,2,Karim,Benali,Lausanne,Switzerland,karim.benali@email.com,2024-02-03
2,4,Lucas,Dubois,Lyon,France,lucas.dubois@email.com,2024-03-25
3,5,Emma,Schneider,Zurich,Switzerland,emma.schneider@email.com,2024-04-01
4,7,Maya,Haddad,Geneva,Switzerland,maya.haddad@email.com,2024-05-07
5,8,Leo,Moreau,Paris,France,leo.moreau@email.com,2024-05-21


In [18]:
# Combining AND and OR (the PARENTHESES are important to make the logic clearer and help avoid mistakes)
query = ("""
SELECT *
FROM Customers
WHERE country = 'Switzerland'
    AND (city = 'Geneva'
    OR city = 'Lausanne');
""")
run_query(query)

,customer_id,first_name,last_name,city,country,email,signup_date
0,1,Alice,Martin,Geneva,Switzerland,alice.martin@email.com,2024-01-15
1,2,Karim,Benali,Lausanne,Switzerland,karim.benali@email.com,2024-02-03
2,7,Maya,Haddad,Geneva,Switzerland,maya.haddad@email.com,2024-05-07


# OPERATOR

## Comparison
- **`=`**   -- equal to
- **`<>`**  -- not equal to
- **`!=`**  -- not equal to (DBMS dependent)
- **`>`**   -- greater than
- **`<`**   -- less than
- **`>=`**  -- greater than or equal to
- **`<=`**  -- less than or equal to


## Logical
- **`AND`** -- returns true if both conditions are true
- **`OR`**  -- returns true if at least one condition is true
- **`NOT`** -- negates a condition


## Arithmetic
- **`+`** -- addition
- **`-`** -- subtraction
- **`*`** -- multiplication
- **`/`** -- division
- **`%`** -- modulo (DBMS dependent)

## Pattern Matching
- **`LIKE`**  -- matches a specified pattern


**Pattern Wildcards**
- **`%`** -- matches zero or more characters
- **`_`** -- matches exactly one character

In [19]:
# LIKE is used for pattern matching in text values
# % -> any sequence of characters (zero, one or multiples characters)
query = ("""
SELECT first_name
FROM Customers
WHERE last_name LIKE 'R%';
""")
run_query(query)

,first_name
0,Sofia


In [20]:
# LIKE is used for pattern matching in text values
# _ -> exactly one character
query = ("""
SELECT first_name
FROM Customers
WHERE last_name LIKE '_o%';
""")
run_query(query)

# _ and % can be combined to put restriction on length (e.g. begin with 'a' and have a minimal length of 3)
# SELECT * FROM Customers
# WHERE CustomerName LIKE 'a__%';

,first_name
0,Sofia
1,Leo


## Range
- **`BETWEEN`**     -- checks if a value is within a range
- **`NOT BETWEEN`** -- checks if a value is outside a range

In [21]:
# BEETWEEN is used to filter values within a range (BOUNDARY INCLUDED)
query = ("""
SELECT unit_price
FROM Products
WHERE unit_price BETWEEN 10 AND 20;
""")
run_query(query)

,unit_price
0,18.5
1,15.0


In [22]:
# Also possible with NOT BETWEEN
query = ("""
SELECT unit_price
FROM Products
WHERE unit_price NOT BETWEEN 10 AND 20;
""")
run_query(query)

,unit_price
0,7.9
1,2.5
2,2.5
3,24.9
4,79.0
5,9.9
6,29.9
7,44.0


In [23]:
# BETWEEN with text values (Alphabetically)
query = ("""
SELECT product_name
FROM Products
WHERE product_name BETWEEN 'Coffee Beans 1kg' AND 'SQL for beginners';
""")
run_query(query)

,product_name
0,Coffee Beans 1kg
1,Green Tea Box
2,Protein Bar
3,Potato Chips
4,Mechanical Keyboard
5,SQL for Beginners
6,Old Headphones


In [24]:
# BETWEEN with date values (Chronogically)
query = ("""
SELECT status FROM Orders
WHERE order_date BETWEEN '2025-01-01' AND '2025-04-01';
""")
run_query(query)

,status
0,shipped
1,pending
2,delivered
3,cancelled
4,delivered
5,shipped
6,pending
7,delivered
8,pending
9,delivered



## Membership
- **`IN`**     -- checks if a value matches any value in a list
- **`NOT IN`** -- checks if a value does not match any value in a list

In [25]:
# IN is used to test whether a value belongs to a list of values (shorthand for multiple OR conditions)
query = ("""
SELECT first_name
FROM Customers
WHERE country IN ('Swiss', 'France', 'Spain')
""")
run_query(query)

,first_name
0,Lucas
1,Noah
2,Leo


In [26]:
# Also possible with NOT IN
query = ("""
SELECT first_name
FROM Customers
WHERE country NOT IN ('Swiss', 'France', 'Spain')
""")
run_query(query)

,first_name
0,Alice
1,Karim
2,Sofia
3,Emma
4,Maya


In [27]:
# IN/NOT IN are very efficient with subqueries
query = ("""
SELECT first_name
FROM Customers
WHERE customer_id NOT IN (SELECT customer_id FROM orders);
""")
run_query(query)

,first_name
0,Leo


## NULL
- **`IS NULL`**     -- checks if a value is NULL
- **`IS NOT NULL`** -- checks if a value is not NULL

SQL has some built-in functions to handle NULL values (default value if null):
- **`COALESCE()`**     -- MYSQL, SQL Server and Oracle
- **`IFNULL()`** -- MYSQL

IFNULL() is specific to MYSQL, otherwise :
- **`ISNULL()`**     -- SQL Server
- **`NVL()`** -- Oracle
- **`IsNull()`**     -- MS Access

In [28]:
# NULL represents a missing or unknown value
# IS NULL is used to find missing values
query = """
SELECT *
FROM customers
WHERE customer_id IS NULL;
"""
run_query(query)

,customer_id,first_name,last_name,city,country,email,signup_date


In [29]:
# NULL represents a missing or unknown value
# IS NOT NULL is used to find non-missing values
query = """
SELECT *
FROM customers
WHERE customer_id IS NOT NULL;
"""
run_query(query)

,customer_id,first_name,last_name,city,country,email,signup_date
0,1,Alice,Martin,Geneva,Switzerland,alice.martin@email.com,2024-01-15
1,2,Karim,Benali,Lausanne,Switzerland,karim.benali@email.com,2024-02-03
2,3,Sofia,Rossi,Milan,Italy,sofia.rossi@email.com,2024-03-11
3,4,Lucas,Dubois,Lyon,France,lucas.dubois@email.com,2024-03-25
4,5,Emma,Schneider,Zurich,Switzerland,emma.schneider@email.com,2024-04-01
5,6,Noah,Garcia,Barcelona,Spain,noah.garcia@email.com,2024-04-18
6,7,Maya,Haddad,Geneva,Switzerland,maya.haddad@email.com,2024-05-07
7,8,Leo,Moreau,Paris,France,leo.moreau@email.com,2024-05-21


In [30]:
# IFNULL(expression, replacement_value)
# IF expression is NULL, return replacement_value, otherwise return expression
query = """
SELECT IFNULL(customer_id, 0)
FROM customers;
"""
run_query(query)

,"IFNULL(customer_id, 0)"
0,1
1,5
2,2
3,8
4,4
5,7
6,6
7,3


In [31]:
# COALESCE(expression1, expression2, ..., [replacement_value])
# Returns the first non-NULL expression in the list
# If only null values, returns NULL or replacement_value
query = """
SELECT first_name, COALESCE(email, city, country, 'no more information')
FROM customers;
"""
run_query(query)

,first_name,"COALESCE(email, city, country, 'no more information')"
0,Alice,alice.martin@email.com
1,Karim,karim.benali@email.com
2,Sofia,sofia.rossi@email.com
3,Lucas,lucas.dubois@email.com
4,Emma,emma.schneider@email.com
5,Noah,noah.garcia@email.com
6,Maya,maya.haddad@email.com
7,Leo,leo.moreau@email.com



## Subquery Operators
- **`EXISTS`** — returns true if the subquery returns at least one row
- **`IN`** — compares a value with any value returned by a subquery
- **`ANY`** — returns true if the comparison is true for at least one value returned by the subquery
- **`ALL`** — returns true if the comparison is true for all values returned by the subquery

In [32]:
# EXISTS is used to check wheter a subquery returns at least one row, and it returns true if it does
# NOT EXISTS is also useful
query = """
SELECT product_name
FROM products AS p
WHERE EXISTS (
    SELECT 1
    FROM order_items AS o
    WHERE p.product_id = o.product_id AND o.unit_price > 15
)
"""

run_query(query)

,product_name
0,Coffee Beans 1kg
1,Wireless Mouse
2,Mechanical Keyboard
3,SQL for Beginners
4,Advanced Data Modeling


In [33]:
# IN compares a value to at least one value returned by a subquery
query = """
SELECT product_name
FROM products
WHERE product_id IN (
    SELECT product_id
    FROM order_items
    WHERE quantity > 3
)
"""

run_query(query)

,product_name
0,Green Tea Box
1,Protein Bar
2,Potato Chips


In [34]:
# ANY is used to compare a value to any value returned by a subquery
# Not available on sqlite, but available on MYSQL, SQL Server and Oracle
query = """
SELECT *
FROM products
WHERE unit_price > ANY (
    SELECT unit_price
    FROM products
    WHERE category_id = 1
)
"""

In [35]:
# ALL is used to compare a value to all values returned by a subquery
# Not available on sqlite, but available on MYSQL, SQL Server and Oracle
query = """
SELECT *
FROM products
WHERE unit_price > ALL (
    SELECT unit_price
    FROM products
    WHERE category_id = 1
)
"""


## Set Operators
- **`UNION`**      -- combines results and removes duplicates
- **`UNION ALL`**  -- combines results and keeps duplicates
- **`INTERSECT`**  -- returns rows present in both queries
- **`EXCEPT`**     -- returns rows from the first query not in the second

In [36]:
# UNION combines results and removes duplicates, rows must be in the same order
query = """
SELECT country
FROM customers
UNION
SELECT country
FROM employees
"""

run_query(query)

,country
0,Algeria
1,France
2,Italy
3,Spain
4,Switzerland


In [37]:
# UNION ALL combines results and keeps duplicates, rows must be in the same order
query = """
SELECT country
FROM customers
UNION ALL
SELECT country
FROM employees
"""

run_query(query)

,country
0,Switzerland
1,Switzerland
2,Italy
3,France
4,Switzerland
5,Spain
6,Switzerland
7,France
8,Switzerland
9,Algeria


In [38]:
# INTERSECT returns rows present in both queries, rows must be in the same order
query = """
SELECT country
FROM customers
INTERSECT
SELECT country
FROM employees
"""

run_query(query)

,country
0,France
1,Switzerland


In [39]:
# EXCEPT returns rows from the first query that are not present in the second query, rows must be in the same order
query = """
SELECT country
FROM customers
EXCEPT
SELECT country
FROM employees
"""

run_query(query)

,country
0,Italy
1,Spain



## String Operators
- **`||`**        -- string concatenation (PostgreSQL, Oracle, sqlite)
- **`CONCAT()`**  -- concatenates strings (MYSQL)

In [40]:
query = """
SELECT first_name || ' ' || last_name AS [Full name 1], CONCAT(first_name, ' ', last_name) AS [Full name 2]
FROM customers;
"""

run_query(query)

,Full name 1,Full name 2
0,Alice Martin,Alice Martin
1,Karim Benali,Karim Benali
2,Sofia Rossi,Sofia Rossi
3,Lucas Dubois,Lucas Dubois
4,Emma Schneider,Emma Schneider
5,Noah Garcia,Noah Garcia
6,Maya Haddad,Maya Haddad
7,Leo Moreau,Leo Moreau


# 3. ORDER BY
The **`ORDER BY`** clause is used to sort the result set of a query (organize rows in ascending or descending order based on one or more columns).



In [41]:
# ORDER BY sort the result in ascending (ASC) order by default (smallest to largest, alphabetically, oldest to newest)
query = ("""
SELECT *
FROM products
ORDER BY unit_price;
""")
run_query(query)

,product_id,product_name,category_id,unit_price,stock_quantity,active
0,3,Protein Bar,12,2.5,300,1
1,4,Potato Chips,2,2.5,150,1
2,2,Green Tea Box,11,7.9,85,1
3,7,USB-C Cable,3,9.9,200,1
4,10,Old Headphones,3,15.0,0,0
5,1,Coffee Beans 1kg,10,18.5,120,1
6,5,Wireless Mouse,3,24.9,40,1
7,8,SQL for Beginners,4,29.9,60,1
8,9,Advanced Data Modeling,4,44.0,20,1
9,6,Mechanical Keyboard,3,79.0,25,1


In [42]:
# ORDER BY sorting in descending (DESC) order
query = ("""
SELECT *
FROM products
ORDER BY unit_price DESC;
""")
run_query(query)

,product_id,product_name,category_id,unit_price,stock_quantity,active
0,6,Mechanical Keyboard,3,79.0,25,1
1,9,Advanced Data Modeling,4,44.0,20,1
2,8,SQL for Beginners,4,29.9,60,1
3,5,Wireless Mouse,3,24.9,40,1
4,1,Coffee Beans 1kg,10,18.5,120,1
5,10,Old Headphones,3,15.0,0,0
6,7,USB-C Cable,3,9.9,200,1
7,2,Green Tea Box,11,7.9,85,1
8,3,Protein Bar,12,2.5,300,1
9,4,Potato Chips,2,2.5,150,1


In [43]:
# For string values, ORDER BY sort alphabetically
query = ("""
SELECT *
FROM products
ORDER BY product_name;
""")
run_query(query)

,product_id,product_name,category_id,unit_price,stock_quantity,active
0,9,Advanced Data Modeling,4,44.0,20,1
1,1,Coffee Beans 1kg,10,18.5,120,1
2,2,Green Tea Box,11,7.9,85,1
3,6,Mechanical Keyboard,3,79.0,25,1
4,10,Old Headphones,3,15.0,0,0
5,4,Potato Chips,2,2.5,150,1
6,3,Protein Bar,12,2.5,300,1
7,8,SQL for Beginners,4,29.9,60,1
8,7,USB-C Cable,3,9.9,200,1
9,5,Wireless Mouse,3,24.9,40,1


In [44]:
# For date values, ORDER BY sort chronogically
query = ("""
SELECT *
FROM orders
ORDER BY order_date;
""")
run_query(query)

,order_id,customer_id,employee_id,order_date,status
0,1,1,1,2025-01-10,shipped
1,2,2,2,2025-01-12,pending
2,3,1,1,2025-01-18,delivered
3,4,3,2,2025-02-02,cancelled
4,5,4,1,2025-02-08,delivered
5,6,5,3,2025-02-18,shipped
6,7,6,2,2025-03-01,pending
7,8,7,1,2025-03-03,delivered
8,9,7,3,2025-03-05,pending
9,10,2,2,2025-03-07,delivered


In [45]:
# ORDER BY with several column (fisrt column first and the second column to break ties)
query = ("""
SELECT *
FROM products
ORDER BY unit_price DESC, product_name ASC;
""")
run_query(query)

,product_id,product_name,category_id,unit_price,stock_quantity,active
0,6,Mechanical Keyboard,3,79.0,25,1
1,9,Advanced Data Modeling,4,44.0,20,1
2,8,SQL for Beginners,4,29.9,60,1
3,5,Wireless Mouse,3,24.9,40,1
4,1,Coffee Beans 1kg,10,18.5,120,1
5,10,Old Headphones,3,15.0,0,0
6,7,USB-C Cable,3,9.9,200,1
7,2,Green Tea Box,11,7.9,85,1
8,4,Potato Chips,2,2.5,150,1
9,3,Protein Bar,12,2.5,300,1


In [46]:
# ORDER BY with a not displayed column
query = ("""
SELECT product_name
FROM products
ORDER BY unit_price DESC;
""")
run_query(query)

,product_name
0,Mechanical Keyboard
1,Advanced Data Modeling
2,SQL for Beginners
3,Wireless Mouse
4,Coffee Beans 1kg
5,Old Headphones
6,USB-C Cable
7,Green Tea Box
8,Protein Bar
9,Potato Chips


In [47]:
# ORDER BY with an alias
query = ("""
SELECT order_id, unit_price * quantity as [Total price]
FROM order_items
ORDER BY [Total price] DESC;
""")
run_query(query)

,order_id,Total price
0,4,79.0
1,10,79.0
2,8,59.8
3,7,49.8
4,6,44.0
5,1,37.0
6,9,31.6
7,3,29.9
8,7,29.7
9,5,25.0


In [48]:
# ORDER BY with an expression
query = ("""
SELECT order_id, unit_price, quantity
FROM order_items
ORDER BY unit_price * quantity DESC;
""")
run_query(query)

,order_id,unit_price,quantity
0,4,79.0,1
1,10,79.0,1
2,8,29.9,2
3,7,24.9,2
4,6,44.0,1
5,1,18.5,2
6,9,7.9,4
7,3,29.9,1
8,7,9.9,3
9,5,2.5,10


In [49]:
# ORDER BY is often combined with LIMIT to return a meaningful subset of rows
query = ("""
SELECT *
FROM products
ORDER BY unit_price DESC
LIMIT 3;
""")
run_query(query)

,product_id,product_name,category_id,unit_price,stock_quantity,active
0,6,Mechanical Keyboard,3,79.0,25,1
1,9,Advanced Data Modeling,4,44.0,20,1
2,8,SQL for Beginners,4,29.9,60,1


# 4. CASE
CASE expression is used to create conditional logic in SQL (return different values depending on one or more conditions)

In [50]:
# Most common form (AS is not mandatory, but better for readability - END is mandatory) - BE CAREFUL: ORDER OF CONDITIONS MATTERS !!
query = """
SELECT product_name, unit_price as price,
  CASE
    WHEN unit_price < 20 THEN "Low Cost"
    WHEN unit_price BETWEEN 20 and 50 THEN "Medium Cost"
    ELSE "High Cost"
  END AS PriceCategory
FROM products
"""

run_query(query)

,product_name,price,PriceCategory
0,Coffee Beans 1kg,18.5,Low Cost
1,Green Tea Box,7.9,Low Cost
2,Protein Bar,2.5,Low Cost
3,Potato Chips,2.5,Low Cost
4,Wireless Mouse,24.9,Medium Cost
5,Mechanical Keyboard,79.0,High Cost
6,USB-C Cable,9.9,Low Cost
7,SQL for Beginners,29.9,Medium Cost
8,Advanced Data Modeling,44.0,Medium Cost
9,Old Headphones,15.0,Low Cost


In [51]:
# This form compares one expression to several possible values (two ways here: OR or IN)
query = """
SELECT order_id,
  CASE
    WHEN status IN ('shipped', 'delivered') THEN "OK"
    WHEN status = 'pending' OR status = 'cancelled' THEN "NOT OK"
    ELSE "IDK"
  END AS status_eval
FROM orders
"""

run_query(query)

,order_id,status_eval
0,1,OK
1,2,NOT OK
2,3,OK
3,4,NOT OK
4,5,OK
5,6,OK
6,7,NOT OK
7,8,OK
8,9,NOT OK
9,10,OK


In [52]:
# CASE can be used in ORDER BY to apply a custome sort order
query = """
SELECT order_id, status
FROM orders
ORDER BY
  CASE
    WHEN status = 'delivered' THEN 1
    WHEN status = 'shipped' THEN 2
    WHEN status = 'pending' THEN 3
    WHEN status = 'cancelled' THEN 4
  END
"""

run_query(query)

,order_id,status
0,3,delivered
1,5,delivered
2,8,delivered
3,10,delivered
4,1,shipped
5,6,shipped
6,2,pending
7,7,pending
8,9,pending
9,4,cancelled


In [53]:
# CASE is often combined with GROUP BY to group rows base on personalized categories
query = """
SELECT
  CASE
    WHEN unit_price < 20 THEN "Low Cost"
    WHEN unit_price BETWEEN 20 and 50 THEN "Medium Cost"
    ELSE "High Cost"
  END AS PriceCategory,
  COUNT(*) as count
FROM products
GROUP BY
  PriceCategory
ORDER BY
  CASE
    WHEN unit_price < 20 THEN 1
    WHEN unit_price BETWEEN 20 and 50 THEN 2
    ELSE 3
  END
"""

run_query(query)

,PriceCategory,count
0,Low Cost,6
1,Medium Cost,3
2,High Cost,1


In [54]:
# CASE can also be used with aggregate functions, for example to do a conditionnal summing
query = """
SELECT
  SUM(CASE WHEN o.status = 'delivered' OR status = 'shipped' THEN oi.unit_price*oi.quantity ELSE 0 END) as OK_amount,
  SUM(CASE WHEN o.status = 'pending' OR status = 'cancelled' THEN oi.unit_price*oi.quantity ELSE 0 END) as NOT_OK_amount
FROM order_items AS oi
JOIN orders AS o
ON o.order_id = oi.order_id
"""

run_query(query)

,OK_amount,NOT_OK_amount
0,372.9,239.8


In [55]:
# Also possible to do SUM(CASE WHEN condition THEN 0 ELSE 1 END) - conditionnal counting
#  PIVOT TABLES withn suming also possible
query = """
SELECT
  SUM(CASE WHEN o.status = 'delivered' OR status = 'shipped' THEN 1 ELSE 0 END) as OK_count,
  SUM(CASE WHEN o.status = 'pending' OR status = 'cancelled' THEN 1 ELSE 0 END) as NOT_OK_count
FROM orders o
"""

run_query(query)

,OK_count,NOT_OK_count
0,6,4


# 5. ALIASES
An alias is a temporary name given to a column or a table in a query.

In [56]:
# ALIASES are used to give a column or table a temporary name (table ALIASES is useful when joining tables and column ALIASES improve result readability)
# AS Keyword is optionnal but it's clearer and more readable with it
query = ("""
SELECT AVG(unit_price) AS avgPrice
FROM products AS prd
""")

run_query(query)

,avgPrice
0,23.41


In [57]:
# ALIASES with spaces (two way [] OR "")
query = ("""
SELECT AVG(unit_price) AS [Average Price], SUM(unit_price) AS "Total Price"
FROM products AS prd
""")

run_query(query)

,Average Price,Total Price
0,23.41,234.1


In [58]:
# ALIASES are also very useful for expression (arithmetic or text expression)
query = ("""
SELECT first_name || ' ' || last_name as [Full name], city || ', ' || country as Location
FROM customers
""")

run_query(query)

,Full name,Location
0,Alice Martin,"Geneva, Switzerland"
1,Karim Benali,"Lausanne, Switzerland"
2,Sofia Rossi,"Milan, Italy"
3,Lucas Dubois,"Lyon, France"
4,Emma Schneider,"Zurich, Switzerland"
5,Noah Garcia,"Barcelona, Spain"
6,Maya Haddad,"Geneva, Switzerland"
7,Leo Moreau,"Paris, France"


In [59]:
# Table ALIASES allow to have eaiser the read JOIN queries
query = """
SELECT o.order_id, SUM(unit_price) as "Total Price"
FROM orders AS o
JOIN order_items AS oi
ON o.order_id = oi.order_id
GROUP BY o.order_id
"""

run_query(query)

,order_id,Total Price
0,1,21.0
1,2,34.8
2,3,37.8
3,4,79.0
4,5,5.0
5,6,62.5
6,7,34.8
7,8,32.4
8,9,10.4
9,10,97.5


In [60]:
# Table ALIASES are essential for self join queries
query = """
SELECT a.first_name || " " || a.last_name AS [Customer name (a)], b.first_name || " " || b.last_name AS [Customer name (b)], a.city
FROM customers AS a, customers AS b
WHERE a.customer_id <> b.customer_id
AND a.city = b.city
ORDER BY a.city
"""

run_query(query)

,Customer name (a),Customer name (b),city
0,Alice Martin,Maya Haddad,Geneva
1,Maya Haddad,Alice Martin,Geneva


In [61]:
# Column ALIASES can be reused in ORDER BY (or GROUP BY, HAVING, etc)
query = """
SELECT order_id, unit_price * quantity as [Total price]
FROM order_items
ORDER BY [Total price] DESC;
"""

run_query(query)

,order_id,Total price
0,4,79.0
1,10,79.0
2,8,59.8
3,7,49.8
4,6,44.0
5,1,37.0
6,9,31.6
7,3,29.9
8,7,29.7
9,5,25.0


# 6. TEXT FUNCTIONS
Text functions are used to manipulate and transform string values in SQL.

In [62]:
# LOWER(text) -> converts text to lowercase
# UPPER(text) -> converts text to uppercase
# TRIM(text) -> removes spaces at the beginning and end
# LENGTH(text) -> returns the number of characters
# SUBSTR(text, start, length) -> extracts part of the text
# REPLACE(text, old_value, new_value) -> replaces one text value with another
query = ("""
SELECT LOWER(product_name), UPPER(product_name), TRIM(product_name), LENGTH(product_name), SUBSTR(product_name, 1, 3), REPLACE(product_name, ' ', '_')
FROM products
""")

run_query(query)

,LOWER(product_name),UPPER(product_name),TRIM(product_name),LENGTH(product_name),"SUBSTR(product_name, 1, 3)","REPLACE(product_name, ' ', '_')"
0,coffee beans 1kg,COFFEE BEANS 1KG,Coffee Beans 1kg,16,Cof,Coffee_Beans_1kg
1,green tea box,GREEN TEA BOX,Green Tea Box,13,Gre,Green_Tea_Box
2,protein bar,PROTEIN BAR,Protein Bar,11,Pro,Protein_Bar
3,potato chips,POTATO CHIPS,Potato Chips,12,Pot,Potato_Chips
4,wireless mouse,WIRELESS MOUSE,Wireless Mouse,14,Wir,Wireless_Mouse
5,mechanical keyboard,MECHANICAL KEYBOARD,Mechanical Keyboard,19,Mec,Mechanical_Keyboard
6,usb-c cable,USB-C CABLE,USB-C Cable,11,USB,USB-C_Cable
7,sql for beginners,SQL FOR BEGINNERS,SQL for Beginners,17,SQL,SQL_for_Beginners
8,advanced data modeling,ADVANCED DATA MODELING,Advanced Data Modeling,22,Adv,Advanced_Data_Modeling
9,old headphones,OLD HEADPHONES,Old Headphones,14,Old,Old_Headphones


In [63]:
# TEXT FUNCTIONS can also be used with WHERE
query = """
SELECT *
FROM products
WHERE LENGTH(product_name) > 15;
"""

run_query(query)

,product_id,product_name,category_id,unit_price,stock_quantity,active
0,1,Coffee Beans 1kg,10,18.5,120,1
1,6,Mechanical Keyboard,3,79.0,25,1
2,8,SQL for Beginners,4,29.9,60,1
3,9,Advanced Data Modeling,4,44.0,20,1


# 7. REGULAR EXPRESSIONS (REGEX)

**Most useful regex symbols :**

- `^` = start of the string
- `$` = end of the string
- `.` = any single character
- `[abc]` = one character from the set
- `[a-z]` = one lowercase letter
- `[A-Z]` = one uppercase letter
- `[0-9]` = one digit
- `+` = one or more occurrences
- `*` = zero or more occurrences
- `{n}` = exactly `n` occurrences

In MySQL, `REGEXP_LIKE(expr, pattern)` is used for advanced pattern matching.

**Common examples** :

- `REGEXP_LIKE(customer_name, '^b')` → matches values that start with `b`
- `REGEXP_LIKE(customer_name, 'fy$')` → matches values that end with `fy`
- `REGEXP_LIKE(product_code, '[0-9]')` → matches values that contain at least one digit
- `REGEXP_LIKE(customer_code, '^[0-9]+$')` → matches values that contain only digits
- `REGEXP_LIKE(customer_name, '^.{5}$')` → matches values with exactly 5 characters
- `REGEXP_LIKE(product_code, '^[A-Z]{3}[0-9]{4}$')` → matches values with 3 uppercase letters followed by 4 digits
- `REGEXP_LIKE(email, '@company\\.com$')` → matches values that end with `@company.com`
- `REGEXP_LIKE(customer_name, '^b', 'c')` → matches values that start with lowercase `b` only (`c` = case-sensitive)

**Important note**

Without `^` and `$`, MySQL checks whether the pattern appears anywhere in the string.

# 8. NUMERIC FUNCTIONS
Numeric functions are used to perform calculations and transformations on numbers in SQL.

In [64]:
# Simple artihmetic operations ("+", "-", "*", "/", "%")
query = """
SELECT unit_price, unit_price + 2, unit_price - 2, unit_price * 2, unit_price / 2, unit_price % 2
FROM products
"""

run_query(query)

,unit_price,unit_price + 2,unit_price - 2,unit_price * 2,unit_price / 2,unit_price % 2
0,18.5,20.5,16.5,37.0,9.25,0.0
1,7.9,9.9,5.9,15.8,3.95,1.0
2,2.5,4.5,0.5,5.0,1.25,0.0
3,2.5,4.5,0.5,5.0,1.25,0.0
4,24.9,26.9,22.9,49.8,12.45,0.0
5,79.0,81.0,77.0,158.0,39.50,1.0
6,9.9,11.9,7.9,19.8,4.95,1.0
7,29.9,31.9,27.9,59.8,14.95,1.0
8,44.0,46.0,42.0,88.0,22.00,0.0
9,15.0,17.0,13.0,30.0,7.50,1.0


In [65]:
# ROUND(number) -> rounds a number to the nearest integer
# ROUND(number, decimal_places) -> rounds a number to a specified number of decimal places
# ABS(number) -> returns the absolute value of a number
# CEIL(number) -> rounds a number up to the next integer
# FLOOR(number) -> rounds a number down to the previous integer
# MOD(number, divisor) -> returns the remainder after division
# POWER(number, exponent) -> raises a number to a power
# SQRT(number) -> returns the square root of a number
query = """
SELECT unit_price, ROUND(unit_price), ABS(unit_price), CEIL(unit_price), FLOOR(unit_price), MOD(unit_price, 2), POWER(unit_price, 2), SQRT(unit_price)
FROM products
"""

run_query(query)

,unit_price,ROUND(unit_price),ABS(unit_price),CEIL(unit_price),FLOOR(unit_price),"MOD(unit_price, 2)","POWER(unit_price, 2)",SQRT(unit_price)
0,18.5,19.0,18.5,19.0,18.0,0.5,342.25,4.301163
1,7.9,8.0,7.9,8.0,7.0,1.9,62.41,2.810694
2,2.5,3.0,2.5,3.0,2.0,0.5,6.25,1.581139
3,2.5,3.0,2.5,3.0,2.0,0.5,6.25,1.581139
4,24.9,25.0,24.9,25.0,24.0,0.9,620.01,4.989990
5,79.0,79.0,79.0,79.0,79.0,1.0,6241.00,8.888194
6,9.9,10.0,9.9,10.0,9.0,1.9,98.01,3.146427
7,29.9,30.0,29.9,30.0,29.0,1.9,894.01,5.468089
8,44.0,44.0,44.0,44.0,44.0,0.0,1936.00,6.633250
9,15.0,15.0,15.0,15.0,15.0,1.0,225.00,3.872983


In [66]:
# Combining numeric function

query = """
SELECT ROUND(SQRT(ABS(unit_price)), 2)
FROM products
"""

run_query(query)

,"ROUND(SQRT(ABS(unit_price)), 2)"
0,4.30
1,2.81
2,1.58
3,1.58
4,4.99
5,8.89
6,3.15
7,5.47
8,6.63
9,3.87


# 9. DATES FUNCTIONS
Date functions are used to work with dates and times in SQL.

In [ ]:
query_sql = """
WITH t AS (
  SELECT
    CAST('2025-03-29 14:35:42' AS DATETIME) AS d,
    CAST('2025-03-29' AS DATE) AS d1,
    CAST('2024-12-15' AS DATE) AS d2,
    '29/03/2025' AS str_date
)
SELECT
  -- date/heure actuelles
  strftime('%Y', d),
  NOW() AS now_result,
  CURDATE() AS curdate_result,
  CURTIME() AS curtime_result,

  -- extraction via DATE_FORMAT (texte)
  DATE_FORMAT(d, '%Y') AS year_part,
  DATE_FORMAT(d, '%m') AS month_part,
  DATE_FORMAT(d, '%d') AS day_part,
  DATE_FORMAT(d, '%H') AS hour_part,
  DATE_FORMAT(d, '%i') AS minute_part,
  DATE_FORMAT(d, '%s') AS second_part,

  -- extraction directe
  DATE(d) AS date_only,
  TIME(d) AS time_only,
  EXTRACT(YEAR FROM d) AS extract_year,
  EXTRACT(MONTH FROM d) AS extract_month,
  EXTRACT(DAY FROM d) AS extract_day,

  -- additions / soustractions
  DATE_ADD(d, INTERVAL 7 DAY) AS date_add_7_days,
  DATE_SUB(d, INTERVAL 1 MONTH) AS date_sub_1_month,

  -- différences
  DATEDIFF(d1, d2) AS datediff_days,
  TIMESTAMPDIFF(DAY, d2, d1) AS tsdiff_days,
  TIMESTAMPDIFF(MONTH, d2, d1) AS tsdiff_months,
  TIMESTAMPDIFF(YEAR, d2, d1) AS tsdiff_years,
  TIMESTAMPDIFF(HOUR, d2, d1) AS tsdiff_hours,

  -- parsing
  STR_TO_DATE(str_date, '%d/%m/%Y') AS str_to_date_result,

  -- fin de mois
  LAST_DAY(d) AS last_day_result,

  -- unix timestamp
  UNIX_TIMESTAMP(d) AS unix_ts,
  FROM_UNIXTIME(UNIX_TIMESTAMP(d)) AS from_unix_ts

FROM t;
"""

run_query(query_sql)

# 10. CAST AND TYPE CONVERSIONS
**`CAST()`** is used to convert a value from one data type to another (useful when data is not stored in the format you need for analysis).

In [68]:
# Casting to INTEGER
query = """
SELECT CAST('475' AS INTEGER), CAST(12.75 AS INTEGER)
"""

run_query(query)

,CAST('475' AS INTEGER),CAST(12.75 AS INTEGER)
0,475,12


In [69]:
# Casting to REAL
query = """
SELECT CAST('12.75' AS REAL), CAST(12 AS REAL)
"""

run_query(query)

,CAST('12.75' AS REAL),CAST(12 AS REAL)
0,12.75,12.0


In [70]:
# Casting to TEXT
query = """
SELECT CAST('12.75' AS TEXT), CAST(12 AS TEXT)
"""

run_query(query)

,CAST('12.75' AS TEXT),CAST(12 AS TEXT)
0,12.75,12


In [71]:
# Casting to NUMERIC
# NUMERIC = converts the value to a numeric type automatically (real or integer when possible)
query = """
SELECT CAST('45.67' AS NUMERIC), CAST('45.0' AS NUMERIC)
"""

run_query(query)

,CAST('45.67' AS NUMERIC),CAST('45.0' AS NUMERIC)
0,45.67,45


In [72]:
# Casting to BLOB
# BLOB = stores raw binary data, not regular text or numbers
query = """
SELECT CAST('ABC' AS BLOB)
"""

run_query(query)

,CAST('ABC' AS BLOB)
0,b'ABC'


# 11. AGGREGATE FUCTIONS

Aggregate functions are used to perform calculations on multiple rows and return a single result (useful when you want to summarize data).

Aggregate functions are often used with GROUP BY. The GROUP BY clause splits the result-set into groups of values and the aggregate function can be used to return a single value for each group.

Without GROUP BY, an aggregate function returns one result for the whole filtered table.

The most commonly used SQL aggregate functions are:

- **`MIN()`** - returns the smallest value of a column
- **`MAX()`** - returns the largest value of a column
- **`COUNT()`** - returns the number of rows in a set
- **`SUM()`** - returns the sum of a numerical column
- **`AVG()`** - returns the average value of a numerical column

Aggregate functions ignore null values, except for COUNT(*).

## MIN()
**`MIN()`** returns the smallest value in a column.

In [ ]:
# Smallest values of multiple columns
query = ("""
SELECT MIN(unit_price), MIN(stock_quantity)
FROM products;
""")

run_query(query)

,MIN(unit_price),MIN(stock_quantity)
0,2.5,0


In [74]:
# Smallest value of a column with a condition (and alias)
query = ("""
SELECT MIN(unit_price) as MinPriceCatOne
FROM products
WHERE category_id = 3;
""")

run_query(query)

,MinPriceCatOne
0,9.9


## MAX()
**`MIN()`** returns the largest value in a column.

In [ ]:
# Largest values of multiple columns
query = ("""
SELECT MAX(unit_price), MAX(stock_quantity)
FROM products;
""")

run_query(query)

,MAX(unit_price),MAX(stock_quantity)
0,79.0,300


In [76]:
# Largest value of a column with a condition (and alias)
query = ("""
SELECT MAX(unit_price) as MaxPriceCatOne
FROM products
WHERE category_id = 2;
""")

run_query(query)

,MaxPriceCatOne
0,2.5


## COUNT()
**`COUNT()`** returns the number of rows.

In [77]:
# Count all row of the table (null values included)
query = ("""
SELECT COUNT(*)
FROM products
""")

run_query(query)

,COUNT(*)
0,10


In [78]:
# Count of all non-null values in a specified column (only rows where product_name is not null)
query = ("""
SELECT COUNT(product_name)
FROM products
""")

run_query(query)

,COUNT(product_name)
0,10


In [79]:
# Count of all the unique and non-null values in a specified column
query = ("""
SELECT COUNT(DISTINCT category_id)
FROM products
""")

run_query(query)

,COUNT(DISTINCT category_id)
0,6


In [80]:
# Always posible to add condition (for all the different count)
query = ("""
SELECT COUNT(*)
FROM products
WHERE category_id = 3;
""")

run_query(query)

,COUNT(*)
0,4


## SUM()
**`SUM()`** returns the total of a numeric column.

In [81]:
# Sum of a column (possible with conditions)
query = ("""
SELECT SUM(unit_price)
FROM products
WHERE category_id = 3;
""")

run_query(query)

,SUM(unit_price)
0,128.8


In [82]:
# Possible to have an expression inside the SUM()
query = ("""
SELECT SUM(unit_price * 10)
FROM products
WHERE category_id = 3;
""")

run_query(query)

,SUM(unit_price * 10)
0,1288.0


## AVG()

In [83]:
# Average of a column (possible with conditions)
query = ("""
SELECT AVG(unit_price)
FROM products
WHERE category_id = 3;
""")

run_query(query)

,AVG(unit_price)
0,32.2


In [84]:
# Possible to have an expression inside the AVG()
query = ("""
SELECT AVG(unit_price * 10)
FROM products
WHERE category_id = 3;
""")


run_query(query)

,AVG(unit_price * 10)
0,322.0


In [85]:
# It is very common to use several aggregate functions in the same query.
query = ("""
SELECT AVG(unit_price) AS AvgPrice, MIN(unit_price) AS MinPrice, MAX(unit_price) AS MaxPrice, COUNT(unit_price) AS CountPrice
FROM products
""")

run_query(query)

,AvgPrice,MinPrice,MaxPrice,CountPrice
0,23.41,2.5,79.0,10


# 12. GROUP BY

The **`GROUP BY`** clause is used to group rows that have the same value in one or more columns.

GROUP BY is almost always used with aggregate fucntons (**`COUNT()`**, **`MIN()`**, **`MAX()`**, **`AVG()`**, **`SUM()`**)

**Important rule**

When using GROUP BY, every selected column must usually be ***included in GROUP BY*** or ***used inside an aggregate function***

In [86]:
# The simplest use of GROUP BY is grouping by a single column (combination with ORDER BY to sort the result)
query = """
SELECT country, COUNT(customer_id) AS [Nb of customers]
FROM customers
GROUP BY country
ORDER BY [Nb of customers] DESC
"""

run_query(query)

,country,Nb of customers
0,Switzerland,4
1,France,2
2,Spain,1
3,Italy,1


In [87]:
# You can group by more than one column (combination with ORDER BY to sort the result)
query = """
SELECT country, city, COUNT(customer_id) AS [Nb of customers]
FROM customers
GROUP BY country, city
ORDER BY [Nb of customers] DESC
"""

run_query(query)

,country,city,Nb of customers
0,Switzerland,Geneva,2
1,France,Lyon,1
2,France,Paris,1
3,Italy,Milan,1
4,Spain,Barcelona,1
5,Switzerland,Lausanne,1
6,Switzerland,Zurich,1


In [88]:
# Multiple aggregate functions in a GROUP BY
query = """
SELECT category_name, SUM(unit_price), MAX(unit_price), MIN(unit_price), AVG(unit_price)
FROM products AS p
JOIN categories AS c
ON p.category_id = c.category_id
GROUP BY category_name
"""

run_query(query)

,category_name,SUM(unit_price),MAX(unit_price),MIN(unit_price),AVG(unit_price)
0,Books,73.9,44.0,29.9,36.95
1,Electronics,128.8,79.0,9.9,32.20
2,Snacks,2.5,2.5,2.5,2.50


In [89]:
# GROUP BY with a JOIN
query = """
SELECT e.first_name || " " || e.last_name AS [Employee name], COUNT(o.order_id) AS [Nb of orders done]
FROM orders AS o
JOIN employees AS e
ON e.employee_id = o.employee_id
GROUP BY [Employee name]
ORDER BY [Nb of orders done] DESC
"""

run_query(query)

,Employee name,Nb of orders done
0,Sarah Dupont,4
1,Nicolas Weber,4
2,Ines Favre,2


In [90]:
# WHERE vs GROUP BY: WHERE filters rows before grouping / GROUP BY groups the remaining rows.
query = """
SELECT category_ID, SUM(unit_price), MAX(unit_price), MIN(unit_price), AVG(unit_price)
FROM products
WHERE unit_price > 30
GROUP BY category_ID
"""

run_query(query)

,category_id,SUM(unit_price),MAX(unit_price),MIN(unit_price),AVG(unit_price)
0,3,79.0,79.0,79.0,79.0
1,4,44.0,44.0,44.0,44.0


# 13. HAVING
The **`HAVING`** clause is used to filter groups after **`GROUP BY`**. While **`WHERE`** filters individual rows, **`HAVING`** filters the grouped result.

In [91]:
# Keep only the countries with 2 or more customers
query = """
SELECT country, COUNT(customer_id) AS [Nb of customers]
FROM customers
GROUP BY country
HAVING [Nb of customers] >= 2
ORDER BY [Nb of customers] DESC
"""

run_query(query)

,country,Nb of customers
0,Switzerland,4
1,France,2


In [92]:
# WHERE vs HAVING: WHERE filters rows before grouping / HAVING filters groups after grouping
# Here, first we remove the product row with unit_price not greater than 30, then calculate the aggragate by cat and keep only those with SUM(unit_price) greater than 50
query = """
SELECT category_ID, SUM(unit_price), MAX(unit_price), MIN(unit_price), AVG(unit_price)
FROM products
WHERE unit_price > 30
GROUP BY category_ID
HAVING SUM(unit_price) > 50
"""

run_query(query)

,category_id,SUM(unit_price),MAX(unit_price),MIN(unit_price),AVG(unit_price)
0,3,79.0,79.0,79.0,79.0


# 14. JOINs

A **`JOIN`** is used to combine rows from two or more tables based on a related column (essential in SQL because data is often stored in separate tables).

- **`(INNER) JOIN`** -- Returns only rows that have matching values in both table
- **`LEFT (OUTER) JOIN`** -- Returns all rows from the left table, and only matched rows from the right table
- **`RIGHT (OUTER) JOIN`** -- Returns all rows from the right table, and only matched rows from the left table
- **`FULL (OUTER) JOIN`** -- Return all rows when there is a match in either the left of right table

## INNER JOIN

The **`JOIN`** (INNER JOIN) returns only rows that have matching values in both tables. **`INNER JOIN`** or only **`JOIN`** are the same.

In [93]:
# INNER JOIN example
query = """
SELECT p.product_name, c.category_name
FROM products as p
JOIN categories as c
ON p.category_id = c.category_id
"""

run_query(query)

,product_name,category_name
0,Potato Chips,Snacks
1,Wireless Mouse,Electronics
2,Mechanical Keyboard,Electronics
3,USB-C Cable,Electronics
4,SQL for Beginners,Books
5,Advanced Data Modeling,Books
6,Old Headphones,Electronics


In [94]:
# INNER JOIN with multiple tables
query = """
SELECT o.order_date, c.first_name || " " || c.last_name AS [Customer name], e.first_name || " " || e.last_name AS [Employee name]
FROM orders AS o
JOIN customers AS c
ON o.customer_id = c.customer_id
JOIN employees AS e
ON o.employee_id = e.employee_id
"""

run_query(query)

,order_date,Customer name,Employee name
0,2025-01-10,Alice Martin,Sarah Dupont
1,2025-01-12,Karim Benali,Nicolas Weber
2,2025-01-18,Alice Martin,Sarah Dupont
3,2025-02-02,Sofia Rossi,Nicolas Weber
4,2025-02-08,Lucas Dubois,Sarah Dupont
5,2025-02-18,Emma Schneider,Ines Favre
6,2025-03-01,Noah Garcia,Nicolas Weber
7,2025-03-03,Maya Haddad,Sarah Dupont
8,2025-03-05,Maya Haddad,Ines Favre
9,2025-03-07,Karim Benali,Nicolas Weber


## LEFT JOIN
The **`LEFT JOIN`** (LEFT OUTER JOIN) returns all rows from the left table (table 1) and only the matched rows from the right table (table 2). **`LEFT OUTER JOIN`** or only **`LEFT JOIN`** are the same.


In [95]:
# LEFT JOIN example
query = """
SELECT p.product_name, c.category_name
FROM products as p
LEFT JOIN categories as c
ON p.category_id = c.category_id
"""

run_query(query)

,product_name,category_name
0,Coffee Beans 1kg,None
1,Green Tea Box,None
2,Protein Bar,None
3,Potato Chips,Snacks
4,Wireless Mouse,Electronics
5,Mechanical Keyboard,Electronics
6,USB-C Cable,Electronics
7,SQL for Beginners,Books
8,Advanced Data Modeling,Books
9,Old Headphones,Electronics


In [96]:
# Possible to add WHERE to filter the NULL values - become an INNER JOIN
query = """
SELECT p.product_name, c.category_name
FROM products as p
LEFT JOIN categories as c
ON p.category_id = c.category_id
WHERE NOT category_name IS NULL
"""

run_query(query)

,product_name,category_name
0,Potato Chips,Snacks
1,Wireless Mouse,Electronics
2,Mechanical Keyboard,Electronics
3,USB-C Cable,Electronics
4,SQL for Beginners,Books
5,Advanced Data Modeling,Books
6,Old Headphones,Electronics


In [97]:
# Also possible add WHERE to reveal the NULL values = missing matches
query = """
SELECT p.product_name, c.category_name
FROM products as p
LEFT JOIN categories as c
ON p.category_id = c.category_id
WHERE category_name IS NULL
"""

run_query(query)

,product_name,category_name
0,Coffee Beans 1kg,None
1,Green Tea Box,None
2,Protein Bar,None


## RIGHT JOIN
The **`RIGHT JOIN`** (RIGHT OUTER JOIN) returns all rows from the right table (table 2) and only the matched rows from the left table (table 1). **`RIGHT OUTER JOIN`** or only **`RIGHT JOIN`** are the same.


In [98]:
# It's the opposite of the LEFT JOIN
query = """
SELECT p.product_name, c.category_name
FROM products as p
RIGHT JOIN categories as c
ON p.category_id = c.category_id
"""

run_query(query)

,product_name,category_name
0,Potato Chips,Snacks
1,Wireless Mouse,Electronics
2,Mechanical Keyboard,Electronics
3,USB-C Cable,Electronics
4,SQL for Beginners,Books
5,Advanced Data Modeling,Books
6,Old Headphones,Electronics
7,None,Beverages
8,None,Test


## FULL JOIN
The **`FULL JOIN`** (FULL OUTER JOIN) returns all rows when there is a match or not. **`FULL OUTER JOIN`** or only **`FULL JOIN`** are the same.

**BE CAREFUL**

**`FULL JOIN`** can return very large result sets !!

In [99]:
# FULL JOIN example
query = """
SELECT p.product_name, c.category_name
FROM products as p
FULL JOIN categories as c
ON p.category_id = c.category_id
"""

run_query(query)

,product_name,category_name
0,Coffee Beans 1kg,None
1,Green Tea Box,None
2,Protein Bar,None
3,Potato Chips,Snacks
4,Wireless Mouse,Electronics
5,Mechanical Keyboard,Electronics
6,USB-C Cable,Electronics
7,SQL for Beginners,Books
8,Advanced Data Modeling,Books
9,Old Headphones,Electronics


## SELF JOIN
A **`SELF JOIN`** is a regular join, but the table is joined with itself.

In [100]:
# SELF JOIN example
query = """
SELECT a.first_name || " " || a.last_name AS [Customer name (a)], b.first_name || " " || b.last_name AS [Customer name (b)], a.city
FROM customers AS a, customers AS b
WHERE a.customer_id <> b.customer_id
AND a.city = b.city
ORDER BY a.city
"""

run_query(query)

,Customer name (a),Customer name (b),city
0,Alice Martin,Maya Haddad,Geneva
1,Maya Haddad,Alice Martin,Geneva


## CROSS JOIN
CROSS JOIN returns the Cartesian product of two tables (every row from the first table is combined with every row from the second table)

In [101]:
# SELF JOIN example
query = """
SELECT p.product_name, c.category_name
FROM products AS p
CROSS JOIN categories AS c
"""

run_query(query)

,product_name,category_name
0,Coffee Beans 1kg,Beverages
1,Coffee Beans 1kg,Snacks
2,Coffee Beans 1kg,Electronics
3,Coffee Beans 1kg,Books
4,Coffee Beans 1kg,Test
5,Green Tea Box,Beverages
6,Green Tea Box,Snacks
7,Green Tea Box,Electronics
8,Green Tea Box,Books
9,Green Tea Box,Test


# 15. SUBQUERIES
A **`SUBQUERY`** is a query written inside another query (used when the result of one query is needed by another query)

A subquery can appear in different parts of a SQL statement, such as **`WHERE`**, **`FROM`** and **`SELECT`**

## SUBQUERY in WHERE
This is one of the most common uses of subqueries.

In [102]:
# SUBQUERY in WHERE example
query = """
SELECT product_name, unit_price
FROM products
WHERE unit_price > (SELECT AVG(unit_price) FROM products)
"""

run_query(query)

,product_name,unit_price
0,Wireless Mouse,24.9
1,Mechanical Keyboard,79.0
2,SQL for Beginners,29.9
3,Advanced Data Modeling,44.0


In [103]:
# SUBQUERY in WHERE returning multiple values (can be used with IN)
query = """
SELECT product_name, unit_price
FROM products
WHERE category_id IN (SELECT category_id FROM categories WHERE category_name IN ('Books', 'Electronics'))
"""

run_query(query)

,product_name,unit_price
0,Wireless Mouse,24.9
1,Mechanical Keyboard,79.0
2,USB-C Cable,9.9
3,SQL for Beginners,29.9
4,Advanced Data Modeling,44.0
5,Old Headphones,15.0


In [104]:
# SUBQUERY in WHERE returning multiple values (used with NOT IN)
query = """
SELECT first_name
FROM customers
WHERE customer_id NOT IN (SELECT customer_id FROM orders)
"""

run_query(query)

,first_name
0,Leo


## SUBQUERY in FROM
A subquery can also be used as a temporary table inside **`FROM`** (useful when you want to build a result in stages).

**IMPORTANT** It is mandatory to give an aliases to the table from the subquery

In [105]:
# SUBQUERY in FROM example
query = """
SELECT category_name, avg_price
FROM (
  SELECT category_name, AVG(unit_price) AS avg_price
  FROM products AS p
  JOIN categories as c
  ON p.category_id = c.category_id
  GROUP BY category_name
) AS category_summary
WHERE avg_price > 30;
"""

run_query(query)

,category_name,avg_price
0,Books,36.95
1,Electronics,32.20


## SUBQUERY in SELECT
A subquery can also appear inside the **`SELECT`** list.

In [106]:
# SUBQUERY in SELECT example (add the overall avg unit_price to each row)
query = """
SELECT product_name, unit_price,
(SELECT AVG(unit_price) FROM products) AS avg_price
FROM products
"""

run_query(query)

,product_name,unit_price,avg_price
0,Coffee Beans 1kg,18.5,23.41
1,Green Tea Box,7.9,23.41
2,Protein Bar,2.5,23.41
3,Potato Chips,2.5,23.41
4,Wireless Mouse,24.9,23.41
5,Mechanical Keyboard,79.0,23.41
6,USB-C Cable,9.9,23.41
7,SQL for Beginners,29.9,23.41
8,Advanced Data Modeling,44.0,23.41
9,Old Headphones,15.0,23.41


## CORRELATED SUBQUERY
A **`CORRELATED SUBQUERY`** depends on a value from the outer query. The inner query is linked to the current row of the outer query (useful when the comparison depends on the current row).

In [107]:
# CORRELATED SUBQUERY example (returns products whose unit_price is above the average price of producst in the same category)
query = """
SELECT product_name, unit_price
FROM products AS p
WHERE unit_price > (
  SELECT AVG(unit_price)
  FROM products AS pp
  WHERE p.category_id = pp.category_id
)
"""

run_query(query)

,product_name,unit_price
0,Mechanical Keyboard,79.0
1,Advanced Data Modeling,44.0


# 16. COMMON TABLE EXPRESSION (CTE)
A **`COMMON TABLE EXPRESSION`** (CTE) is a temporary named result set created with the **`WITH`** clause. It allows you to define a query first, give it a name, and the use it in the main query (helps break complex queries into smaller and more readable steps).

**Why a CTE can be better than a Subquery**

A CTE and a subquery can often solve the same problem. The difference is that a CTE is usually easier to read because:

- it has a name

- it separates the logic into steps

- it avoids deeply nested queries

This makes complex SQL more understandable.

In [108]:
# COMMON TABLE EXPRESSION example
query = """
WITH high_price_products AS (
  SELECT product_name, unit_price
  FROM products
  WHERE unit_price > 50
)
SELECT *
FROM high_price_products
"""

run_query(query)

,product_name,unit_price
0,Mechanical Keyboard,79.0


In [109]:
# COMMON TABLE EXPRESSION with aggregation
query = """
WITH category_summary AS (
    SELECT category_name, AVG(unit_price) AS avg_price
    FROM products AS p
    JOIN categories AS c
    ON  p.category_id = c.category_id
    GROUP BY category_name
)
SELECT *
FROM category_summary
"""

run_query(query)

,category_name,avg_price
0,Books,36.95
1,Electronics,32.20
2,Snacks,2.50


In [110]:
# A COMMON TABLE EXPRESSION can be used as the source of a larger query
query = """
WITH category_summary AS (
    SELECT category_name, AVG(unit_price) AS avg_price
    FROM products AS p
    JOIN categories AS c
    ON  p.category_id = c.category_id
    GROUP BY category_name
)
SELECT category_name
FROM category_summary
WHERE avg_price < 20
"""

run_query(query)

,category_name
0,Snacks


In [111]:
# Multiple CTE in the same query
query = """
WITH category_summary AS (
    SELECT category_name, AVG(unit_price) AS avg_price
    FROM products AS p
    JOIN categories AS c
    ON  p.category_id = c.category_id
    GROUP BY category_name
),
high_price_categories AS (
  SELECT category_name
  FROM category_summary
  WHERE avg_price < 20
)
SELECT *
FROM high_price_categories;
"""

run_query(query)

,category_name
0,Snacks


In [112]:
# More readable JOIN with CTE
query = """
WITH category_summary AS (
    SELECT category_id, AVG(unit_price) AS avg_price
    FROM products
    GROUP BY category_id
)
SELECT c.category_name, cs.avg_price
FROM category_summary AS cs
JOIN categories AS c
ON cs.category_id = c.category_id
"""

run_query(query)

,category_name,avg_price
0,Snacks,2.50
1,Electronics,32.20
2,Books,36.95


In [113]:
# CTE is very useful when a query naturally has several logical steps
query = """
WITH monthly_orders AS (
    SELECT STRFTIME('%Y-%m', order_date) AS year_month, COUNT(*) AS number_of_orders
    FROM orders
    GROUP BY year_month
)
SELECT *
FROM monthly_orders
ORDER BY year_month
"""

run_query(query)

,year_month,number_of_orders
0,2025-01,3
1,2025-02,3
2,2025-03,4


# 17. WINDOW FUNCTIONS
Window functions perform calculations across a set of rows related to the current row.

Unlike aggregate functions, they do not reduce the result to one row per group.
They return a value for each row while still looking at other rows in the same result set.

**Aggregate Functions vs Window Functions**

- Aggregate functions return one result per group

- Window functions return one result per row according to a specific group

## AGGREGATE VS WINDOW FUNCTIONS

In [114]:
# Difference between AGGREGATE function and WINDOWS FUNCTIONS - AGGREGATE RETURNS ONE ROW PER CATEGORY
query = """
SELECT category_id, AVG(unit_price) AS avg_price
FROM products
GROUP BY category_id
"""

run_query(query)

,category_id,avg_price
0,2,2.50
1,3,32.20
2,4,36.95
3,10,18.50
4,11,7.90
5,12,2.50


In [115]:
# Difference between AGGREGATE function and WINDOWS FUNCTIONS - WINDOWS RETURNS ONE ROW PER PRODUCT, WHILE ALSO SHOWING AVERAGE PRICE OF PRODUCT'S CATEGORY
query = """
SELECT product_name, category_id, AVG(unit_price) OVER (PARTITION BY category_id) AS avg_price
FROM products
"""

run_query(query)

,product_name,category_id,avg_price
0,Potato Chips,2,2.50
1,Wireless Mouse,3,32.20
2,Mechanical Keyboard,3,32.20
3,USB-C Cable,3,32.20
4,Old Headphones,3,32.20
5,SQL for Beginners,4,36.95
6,Advanced Data Modeling,4,36.95
7,Coffee Beans 1kg,10,18.50
8,Green Tea Box,11,7.90
9,Protein Bar,12,2.50


## SYNTAX

```sql
SELECT column_names,
       window_function() OVER (
           PARTITION BY partition_column
           ORDER BY sort_column
           frame_clause
       ) AS alias
FROM table_name;
```

- **`window_function()`** : The function that performs the calculation.
- **`OVER()`** : Specifies how the function is applied.
- **`PARTITION BY`** : *(Optional)* Divides rows into groups before applying the function.
- **`ORDER BY`** : *(Optional)* Sorts rows within each partition.



In [116]:
# WINDOW FUNCTION without PARTITION BY and ORDER BY (the same can be done with SELECT SUBQUERY)
query = """
SELECT product_name, unit_price,
AVG(unit_price) OVER () AS avg_price
FROM products
"""

run_query(query)

,product_name,unit_price,avg_price
0,Coffee Beans 1kg,18.5,23.41
1,Green Tea Box,7.9,23.41
2,Protein Bar,2.5,23.41
3,Potato Chips,2.5,23.41
4,Wireless Mouse,24.9,23.41
5,Mechanical Keyboard,79.0,23.41
6,USB-C Cable,9.9,23.41
7,SQL for Beginners,29.9,23.41
8,Advanced Data Modeling,44.0,23.41
9,Old Headphones,15.0,23.41


In [117]:
# WINDOW FUNCTION with PARTITION BY - The calculation is done separately by the group created by PARTITION BY
query = """
SELECT product_name, unit_price,
AVG(unit_price) OVER (PARTITION BY category_id) AS avg_price
FROM products
"""

run_query(query)

,product_name,unit_price,avg_price
0,Potato Chips,2.5,2.50
1,Wireless Mouse,24.9,32.20
2,Mechanical Keyboard,79.0,32.20
3,USB-C Cable,9.9,32.20
4,Old Headphones,15.0,32.20
5,SQL for Beginners,29.9,36.95
6,Advanced Data Modeling,44.0,36.95
7,Coffee Beans 1kg,18.5,18.50
8,Green Tea Box,7.9,7.90
9,Protein Bar,2.5,2.50


In [118]:
# WINDOW FUNCTION with ORDER BY - Cumulative SUM
query = """
SELECT orders.order_date, unit_price,
sum(unit_price) OVER (ORDER BY orders.order_date) AS sum_price
FROM order_items
JOIN orders
ON order_items.order_id = orders.order_id
"""

run_query(query)

,order_date,unit_price,sum_price
0,2025-01-10,18.5,21.0
1,2025-01-10,2.5,21.0
2,2025-01-12,24.9,55.8
3,2025-01-12,9.9,55.8
4,2025-01-18,29.9,93.6
5,2025-01-18,7.9,93.6
6,2025-02-02,79.0,172.6
7,2025-02-08,2.5,177.6
8,2025-02-08,2.5,177.6
9,2025-02-18,44.0,240.1


In [119]:
# Same but better than last one
query = """
WITH test AS(
    SELECT o.order_date, SUM(oi.unit_price) AS total_price
    FROM order_items oi
    JOIN orders o
    ON oi.order_id = o.order_id
    GROUP BY o.order_date
)
SELECT order_date,
sum(total_price) OVER (ORDER BY order_date) AS sum_price
FROM test
"""

run_query(query)

,order_date,sum_price
0,2025-01-10,21.0
1,2025-01-12,55.8
2,2025-01-18,93.6
3,2025-02-02,172.6
4,2025-02-08,177.6
5,2025-02-18,240.1
6,2025-03-01,274.9
7,2025-03-03,307.3
8,2025-03-05,317.7
9,2025-03-07,415.2


## RANKING FUNCTIONS
- **`ROW_NUMBER()`** = assigns a unique row number to each row inside the window
- **`RANK()`** = assigns the same rank for ties, but leaving gaps after ties
- **`DENSE_RANK()`** = assigns the same rank for ties, but without leaving gaps after ties
- **`PERCENT_RANK()`** = assign a rank between 0 and 1.

In [120]:
# ROW_NUMBER()
query = """
WITH test AS(
    SELECT o.order_date, SUM(oi.unit_price) AS total_price
    FROM order_items oi
    JOIN orders o
    ON oi.order_id = o.order_id
    GROUP BY o.order_date
),
testt AS(
SELECT order_date, total_price,
sum(total_price) OVER (ORDER BY total_price DESC) AS cumulative_sum_price
FROM test
)
SELECT order_date, total_price, cumulative_sum_price,
ROW_NUMBER() OVER (ORDER BY total_price DESC) AS rank
FROM testt
"""

run_query(query)

,order_date,total_price,cumulative_sum_price,rank
0,2025-03-07,97.5,97.5,1
1,2025-02-02,79.0,176.5,2
2,2025-02-18,62.5,239.0,3
3,2025-01-18,37.8,276.8,4
4,2025-01-12,34.8,346.4,5
5,2025-03-01,34.8,346.4,6
6,2025-03-03,32.4,378.8,7
7,2025-01-10,21.0,399.8,8
8,2025-03-05,10.4,410.2,9
9,2025-02-08,5.0,415.2,10


In [121]:
# RANK()
query = """
WITH test AS(
    SELECT o.order_date, SUM(oi.unit_price) AS total_price
    FROM order_items oi
    JOIN orders o
    ON oi.order_id = o.order_id
    GROUP BY o.order_date
),
testt AS(
SELECT order_date, total_price,
sum(total_price) OVER (ORDER BY total_price DESC) AS cumulative_sum_price
FROM test
)
SELECT order_date, total_price, cumulative_sum_price,
RANK() OVER (ORDER BY total_price DESC) AS rank
FROM testt
"""

run_query(query)

,order_date,total_price,cumulative_sum_price,rank
0,2025-03-07,97.5,97.5,1
1,2025-02-02,79.0,176.5,2
2,2025-02-18,62.5,239.0,3
3,2025-01-18,37.8,276.8,4
4,2025-01-12,34.8,346.4,5
5,2025-03-01,34.8,346.4,5
6,2025-03-03,32.4,378.8,7
7,2025-01-10,21.0,399.8,8
8,2025-03-05,10.4,410.2,9
9,2025-02-08,5.0,415.2,10


In [122]:
# RANK()
query = """
WITH test AS(
    SELECT o.order_date, SUM(oi.unit_price) AS total_price
    FROM order_items oi
    JOIN orders o
    ON oi.order_id = o.order_id
    GROUP BY o.order_date
),
testt AS(
SELECT order_date, total_price,
sum(total_price) OVER (ORDER BY total_price DESC) AS cumulative_sum_price
FROM test
)
SELECT order_date, total_price, cumulative_sum_price,
DENSE_RANK() OVER (ORDER BY total_price DESC) AS rank
FROM testt
"""

run_query(query)

,order_date,total_price,cumulative_sum_price,rank
0,2025-03-07,97.5,97.5,1
1,2025-02-02,79.0,176.5,2
2,2025-02-18,62.5,239.0,3
3,2025-01-18,37.8,276.8,4
4,2025-01-12,34.8,346.4,5
5,2025-03-01,34.8,346.4,5
6,2025-03-03,32.4,378.8,6
7,2025-01-10,21.0,399.8,7
8,2025-03-05,10.4,410.2,8
9,2025-02-08,5.0,415.2,9


In [123]:
# PERCENT_RANK()
query = """
WITH test AS(
    SELECT o.order_date, SUM(oi.unit_price) AS total_price
    FROM order_items oi
    JOIN orders o
    ON oi.order_id = o.order_id
    GROUP BY o.order_date
),
testt AS(
SELECT order_date, total_price,
PERCENT_RANK() OVER (ORDER BY total_price DESC) AS percent_rank
FROM test
)
SELECT order_date, total_price, percent_rank
FROM testt
"""

run_query(query)




,order_date,total_price,percent_rank
0,2025-03-07,97.5,0.000000
1,2025-02-02,79.0,0.111111
2,2025-02-18,62.5,0.222222
3,2025-01-18,37.8,0.333333
4,2025-01-12,34.8,0.444444
5,2025-03-01,34.8,0.444444
6,2025-03-03,32.4,0.666667
7,2025-01-10,21.0,0.777778
8,2025-03-05,10.4,0.888889
9,2025-02-08,5.0,1.000000


## VALUE FUNCTIONS
- **`LAG()`** = returns the value of the previous row
- **`LEAD()`** = returns the value of the previous row


In [124]:
# LAG()
query = """
WITH test AS(
    SELECT o.order_date, SUM(oi.unit_price) AS total_price
    FROM order_items oi
    JOIN orders o
    ON oi.order_id = o.order_id
    GROUP BY o.order_date
),
testt AS(
SELECT order_date, total_price,
sum(total_price) OVER (ORDER BY total_price DESC) AS cumulative_sum_price
FROM test
)
SELECT order_date, total_price, cumulative_sum_price,
LAG(total_price) OVER (ORDER BY total_price DESC) AS prev,
total_price - LAG(total_price) OVER (ORDER BY total_price DESC) AS diff
FROM testt
"""

run_query(query)

,order_date,total_price,cumulative_sum_price,prev,diff
0,2025-03-07,97.5,97.5,NaN,NaN
1,2025-02-02,79.0,176.5,97.5,-18.5
2,2025-02-18,62.5,239.0,79.0,-16.5
3,2025-01-18,37.8,276.8,62.5,-24.7
4,2025-01-12,34.8,346.4,37.8,-3.0
5,2025-03-01,34.8,346.4,34.8,0.0
6,2025-03-03,32.4,378.8,34.8,-2.4
7,2025-01-10,21.0,399.8,32.4,-11.4
8,2025-03-05,10.4,410.2,21.0,-10.6
9,2025-02-08,5.0,415.2,10.4,-5.4


In [ ]:
# LEAD()
query = """
WITH test AS(
    SELECT o.order_date, SUM(oi.unit_price) AS total_price
    FROM order_items oi
    JOIN orders o
    ON oi.order_id = o.order_id
    GROUP BY o.order_date
),
testt AS(
SELECT order_date, total_price,
SUM(total_price) OVER (ORDER BY total_price DESC) AS cumulative_sum_price
FROM test
)
SELECT order_date, total_price, cumulative_sum_price,
LEAD(total_price) OVER (ORDER BY total_price DESC) AS next,
total_price - LEAD(total_price) OVER (ORDER BY total_price DESC) as diff
FROM testt
"""

run_query(query)

,order_date,total_price,cumulative_sum_price,next,diff
0,2025-03-07,97.5,97.5,79.0,18.5
1,2025-02-02,79.0,176.5,62.5,16.5
2,2025-02-18,62.5,239.0,37.8,24.7
3,2025-01-18,37.8,276.8,34.8,3.0
4,2025-01-12,34.8,346.4,34.8,0.0
5,2025-03-01,34.8,346.4,32.4,2.4
6,2025-03-03,32.4,378.8,21.0,11.4
7,2025-01-10,21.0,399.8,10.4,10.6
8,2025-03-05,10.4,410.2,5.0,5.4
9,2025-02-08,5.0,415.2,NaN,NaN



## AGGREGATE WINDOW FUNCTIONS
- **`SUM()`** OVER (...)
- **`AVG()`** OVER (...)
- **`COUNT()`** OVER (...)
- **`MIN()`** OVER (...)
- **`MAX()`** OVER (...)

## MOOVING WINDOWS

A moving window uses a frame clause to define which rows around the current row are included, useful for:
- moving averages
- rolling sums
- smoothing trends

In [126]:
# Moving average over 3 rows
query = """
WITH test AS(
    SELECT o.order_date, SUM(oi.unit_price) AS total_price
    FROM order_items oi
    JOIN orders o
    ON oi.order_id = o.order_id
    GROUP BY o.order_date
)
SELECT order_date, total_price,
AVG(total_price) OVER (
    ORDER BY total_price DESC
    ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS moving
FROM test
"""

run_query(query)

,order_date,total_price,moving
0,2025-03-07,97.5,97.500000
1,2025-02-02,79.0,88.250000
2,2025-02-18,62.5,79.666667
3,2025-01-18,37.8,59.766667
4,2025-01-12,34.8,45.033333
5,2025-03-01,34.8,35.800000
6,2025-03-03,32.4,34.000000
7,2025-01-10,21.0,29.400000
8,2025-03-05,10.4,21.266667
9,2025-02-08,5.0,12.133333


In [127]:
# Moving sum over 5 rows
query = """
WITH test AS(
    SELECT o.order_date, SUM(oi.unit_price) AS total_price
    FROM order_items oi
    JOIN orders o
    ON oi.order_id = o.order_id
    GROUP BY o.order_date
)
SELECT order_date, total_price,
SUM(total_price) OVER (
    ORDER BY total_price DESC
    ROWS BETWEEN CURRENT ROW AND 4 FOLLOWING
    ) AS moving
FROM test
"""

run_query(query)

,order_date,total_price,moving
0,2025-03-07,97.5,311.6
1,2025-02-02,79.0,248.9
2,2025-02-18,62.5,202.3
3,2025-01-18,37.8,160.8
4,2025-01-12,34.8,133.4
5,2025-03-01,34.8,103.6
6,2025-03-03,32.4,68.8
7,2025-01-10,21.0,36.4
8,2025-03-05,10.4,15.4
9,2025-02-08,5.0,5.0


In [128]:
# Centered moving average
query = """
WITH test AS(
    SELECT o.order_date, SUM(oi.unit_price) AS total_price
    FROM order_items oi
    JOIN orders o
    ON oi.order_id = o.order_id
    GROUP BY o.order_date
)
SELECT order_date, total_price,
AVG(total_price) OVER (
    ORDER BY total_price DESC
    ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING
    ) AS moving
FROM test
"""

run_query(query)

,order_date,total_price,moving
0,2025-03-07,97.5,88.250000
1,2025-02-02,79.0,79.666667
2,2025-02-18,62.5,59.766667
3,2025-01-18,37.8,45.033333
4,2025-01-12,34.8,35.800000
5,2025-03-01,34.8,34.000000
6,2025-03-03,32.4,29.400000
7,2025-01-10,21.0,21.266667
8,2025-03-05,10.4,12.133333
9,2025-02-08,5.0,7.700000
